# 01 · 환경 설정과 데이터 준비

**이 노트북은 팀에서 한 명만 끝까지 돌리면 된다.** 만들어진 파생 캐시를 Drive 에 올려두면
나머지 팀원은 마지막 셀의 "캐시 내려받기"만 실행하고 바로 실험으로 넘어간다.

| 단계 | 산출물 | 소요(코랩) | 장치 |
|---|---|---|---|
| step1 | `features.parquet` (174MB) | 6~12분 | CPU |
| step2 | `X98.parquet` (175MB) | 4~8분 | CPU |
| step3 | `aligned.parquet` (52MB) | 5~10분 | CPU |
| step4 | `tm5.parquet` (5MB) | 10~20분 | CPU |
| step5 | `oof_comp.parquet` (29MB) | 10~15분 | **GPU** |

전체 40~65분. 캐시를 받아 쓰면 3~6분.

> **메모리 주의** — 무료 티어 RAM 약 12GB 에서 `train.csv` 1,475,092행 처리가 빠듯하다.
> step 을 나눠 돌리고 사이에 런타임을 재시작하면 안전하다. 고용량 런타임이 있으면 그걸 써라.

## 1. GPU / 런타임 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'GPU 없음 — 런타임 유형을 GPU 로 바꿔라'
import psutil, os
print(f'RAM {psutil.virtual_memory().total/2**30:.1f}GB, CPU {os.cpu_count()}코어')

## 2. 저장소 받기 + 의존성

In [ ]:
REPO_URL = 'https://github.com/hyunku9566/lga_data.git'

import os
if os.path.exists('/content/lga-repo'):
    !cd /content/lga-repo && git pull -q
else:
    !git clone -q $REPO_URL /content/lga-repo

# 코랩 전용 requirements. numpy/pandas/torch 는 코랩에 깔린 것을 그대로 쓴다.
# (평가서버용 requirements.txt 의 핀은 Python 3.11 기준이라 코랩에서 빌드 실패한다)
!pip install -q -r /content/lga-repo/requirements-colab.txt

import xgboost, lightgbm, catboost, numpy, pandas
print(f'xgboost  {xgboost.__version__}')
print(f'lightgbm {lightgbm.__version__}')
print(f'catboost {catboost.__version__}')
print(f'numpy    {numpy.__version__}')
print(f'pandas   {pandas.__version__}')


In [ ]:
# GPU 로 실제 학습이 되는지 확인 (여기서 막히면 런타임 유형을 T4 GPU 로 바꿔라)
import numpy as np, xgboost as xgb
X = np.random.rand(2000, 10).astype(np.float32)
y = (X[:, 0] + np.random.rand(2000) * 0.5 > 0.75).astype(int)
try:
    xgb.XGBClassifier(n_estimators=10, device='cuda:0', tree_method='hist',
                      verbosity=0).fit(X, y)
    print('✅ XGBoost GPU 정상')
except Exception as e:
    print(f'⚠️  GPU 사용 불가 → CPU 로 진행한다 ({type(e).__name__})')
    print('   런타임 > 런타임 유형 변경 > T4 GPU 로 바꾸면 10배 빨라진다')
    import os; os.environ['LGA_DEV'] = 'cpu'


## 3. Drive 연결과 경로

`DRIVE_ROOT` 아래에 팀 공유 폴더를 하나 두고 그 안에 `data/`, `cache/`, `ledger/` 를 만든다.
`ledger/` 는 팀원들이 실험 결과를 모으는 곳이라 **공유 설정을 켜둬야** 한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/lga'    # 팀 공유 폴더로 바꿔라
import os
os.environ['LGA_ROOT']    = '/content/lga'
os.environ['LGA_DATA']    = f'{DRIVE_ROOT}/data/'      # 원본 CSV (Drive)
os.environ['LGA_CACHE']   = '/content/lga/cache/'      # 파생 캐시 (로컬 = 빠름)
os.environ['LGA_LEDGER']  = f'{DRIVE_ROOT}/ledger/'    # 실험 원장 (Drive 공유)
os.environ['LGA_ASSETS']  = '/content/lga-repo/assets/'
os.environ['LGA_DEV']     = 'cuda:0'

for d in [f'{DRIVE_ROOT}/data', f'{DRIVE_ROOT}/cache', f'{DRIVE_ROOT}/ledger', '/content/lga/cache']:
    os.makedirs(d, exist_ok=True)

import sys; sys.path.insert(0, '/content/lga-repo/src')
import config as C
print(C.describe())

### 데이터 확보

**로컬 → Drive → 데이터 서버** 순으로 찾아서 없는 것만 받는다.
서버에서 받은 파일은 **Drive 에 자동 백업**되므로, 다음에 다른 노트북을 열어도
(런타임이 바뀌어도) 다시 받지 않는다.

| 상황 | 걸리는 시간 |
|---|---|
| 처음 (서버에서 전부) | 3~6분 |
| 두 번째부터 (Drive 에서) | 30초~2분 |

아이디는 `team`, 비밀번호는 팀 채널에서 받는다. **다른 텍스트를 붙여넣지 마라.**


In [ ]:
# ── 데이터 확보 ───────────────────────────────────────────────
# 로컬 → Drive → 데이터 서버 순으로 찾는다. 서버에서 받은 건 Drive 에 백업하므로
# 다음에 다른 노트북(다른 런타임)을 열어도 다시 받지 않는다.
import os, sys
sys.path.insert(0, '/content/lga-repo')
from getpass import getpass
os.environ['LGA_PASSWORD'] = getpass('team 비밀번호: ').strip()

from src.bootstrap import setup
C = setup(need_optional=True)     # 트랙맨 원본까지 전부 (파생 캐시를 직접 만들 경우 필요)


## 4. 캐시 내려받기 (있으면 여기서 끝)

다른 팀원이 이미 만들어 Drive 에 올려둔 캐시가 있으면 복사만 하면 된다. **3~6분.**

In [ ]:
import shutil, os, glob
src = f'{DRIVE_ROOT}/cache'
got = 0
for f in ['features.parquet','X98.parquet','aligned.parquet','tm5.parquet','oof_comp.parquet']:
    s = os.path.join(src, f)
    if os.path.exists(s):
        d = os.path.join('/content/lga/cache', f)
        if not os.path.exists(d):
            shutil.copy(s, d)
        got += 1
        print(f'{f:22s} {os.path.getsize(d)/2**20:7.1f}MB')
print(f'\n캐시 {got}/5 확보')
if got == 5:
    print('=> 5~7 단계를 건너뛰고 02/03 노트북으로 넘어가라')

## 5. 파생 캐시 생성

캐시가 없을 때만 실행한다. `--steps` 로 나눠 돌릴 수 있고, 이미 있는 단계는 자동으로 건너뛴다.

**RAM 이 부족하면** step 을 1,2 / 3,4 / 5 로 나눠 돌리고 사이에 `런타임 → 세션 다시 시작` 을 해라.

In [ ]:
!cd /content/lga-repo/src && python prepare_data.py --steps 1,2

In [ ]:
!cd /content/lga-repo/src && python prepare_data.py --steps 3,4

In [ ]:
# step5 는 GPU 를 쓴다 (XGBoost 16회 학습)
!cd /content/lga-repo/src && python prepare_data.py --steps 5

## 6. 캐시를 Drive 에 올려 팀과 공유

In [ ]:
import shutil, os
os.makedirs(f'{DRIVE_ROOT}/cache', exist_ok=True)
for f in ['features.parquet','X98.parquet','aligned.parquet','tm5.parquet','oof_comp.parquet']:
    s = os.path.join('/content/lga/cache', f)
    if os.path.exists(s):
        shutil.copy(s, f'{DRIVE_ROOT}/cache/{f}')
        print(f'업로드 {f:22s} {os.path.getsize(s)/2**20:7.1f}MB')
print('\n총 435MB. Drive 업로드는 5~15분 걸린다.')

## 7. 기준선 계산 (팀에서 한 번만)

모든 아이디어는 **하나의 고정 기준선**과 비교된다. 시드 5개로 폴드2024/2023 을 재고
평균과 **표준편차**를 저장한다. 표준편차가 있어야 "이 차이가 노이즈인가"를 판정할 수 있다.

10~20분 걸리고, 결과는 `ledger/baseline.json` 으로 Drive 에 남아 팀원 전체가 공유한다.

In [ ]:
import sys; sys.path.insert(0, '/content/lga-repo/src')
import experiment as E
base = E.get_baseline()
print()
print(f"기준선  폴드2024 {base['m24']:.1f} ±{base['m24_sd']:.1f}")
print(f"        폴드2023 {base['m23']:.1f} ±{base['m23_sd']:.1f}")
print(f"        시드 {base['seeds']}, 피처 {base['nfeat']}")

---
준비 끝. **`03_run_experiments.ipynb`** 로 넘어가라.